<a href="https://colab.research.google.com/github/chintu4/medbot/blob/main/medbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
#Links of Medical files
file1="https://bookchapter.org/kitaplar/Medical%20Diagnosis%20and%20Treatment%20Methods%20in%20Basic%20Medical%20Sciences.pdf"
file2="https://applications.emro.who.int/dsaf/dsa236.pdf"
file3="https://www.dhhs.nh.gov/sites/g/files/ehbemt476/files/documents/2021-11/disease-handbook-complete.pdf"
file4="https://www.dhhs.nh.gov/sites/g/files/ehbemt476/files/documents/2021-11/disease-handbook-complete.pdf"
file5="https://aiimsrishikesh.edu.in/documents/standard-treatment-guidelines.pdf"
file6="https://www.fao.org/4/y1238e/y1238e06.pdf"

urls=[file1,file2,file3,file4,file5,file6]


In [37]:
import os
import requests
import shutil

# Clear and recreate the directory to ensure no partial files remain
if os.path.exists('medical_docs'):
    shutil.rmtree('medical_docs')
os.makedirs('medical_docs', exist_ok=True)

# Adding a User-Agent header to mimic a web browser
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

for i, url in enumerate(urls):
    try:
        print(f"Attempting to download: {url}")
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            with open(f'medical_docs/doc_{i}.pdf', 'wb') as f:
                f.write(response.content)
            print(f"Successfully downloaded doc_{i}.pdf")
        else:
            print(f"Failed to download {url} - Status Code: {response.status_code}")
    except Exception as e:
        print(f"Error downloading {url}: {e}")

Attempting to download: https://bookchapter.org/kitaplar/Medical%20Diagnosis%20and%20Treatment%20Methods%20in%20Basic%20Medical%20Sciences.pdf
Successfully downloaded doc_0.pdf
Attempting to download: https://applications.emro.who.int/dsaf/dsa236.pdf
Successfully downloaded doc_1.pdf
Attempting to download: https://www.dhhs.nh.gov/sites/g/files/ehbemt476/files/documents/2021-11/disease-handbook-complete.pdf
Failed to download https://www.dhhs.nh.gov/sites/g/files/ehbemt476/files/documents/2021-11/disease-handbook-complete.pdf - Status Code: 403
Attempting to download: https://www.dhhs.nh.gov/sites/g/files/ehbemt476/files/documents/2021-11/disease-handbook-complete.pdf
Failed to download https://www.dhhs.nh.gov/sites/g/files/ehbemt476/files/documents/2021-11/disease-handbook-complete.pdf - Status Code: 403
Attempting to download: https://aiimsrishikesh.edu.in/documents/standard-treatment-guidelines.pdf
Successfully downloaded doc_4.pdf
Attempting to download: https://www.fao.org/4/y1238

In [38]:
!pip install -q langchain langchain-community sentence-transformers chromadb pypdf

In [39]:
import os
import warnings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

warnings.filterwarnings("ignore")

# 1. Load PDFs
all_docs = []
doc_folder = "medical_docs/"

print("Processing Documents...")
for file in sorted(os.listdir(doc_folder)):
    if file.endswith(".pdf"):
        try:
            loader = PyMuPDFLoader(os.path.join(doc_folder, file))
            pages = loader.load()
            all_docs.extend(pages)
            print(f"Loaded {file}")
        except Exception as e:
            print(f"Skipping {file}: {e}")

# 2. Create Chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
splits = text_splitter.split_documents(all_docs)

# 3. Create Vector Store (Using In-Memory to avoid SQLite lock issues)
print("Creating Vector Store in-memory...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="medical_assistant_memory"
)

print(f"\nSUCCESS: Vector store created with {len(splits)} chunks.")

Processing Documents...
Loaded doc_0.pdf
Loaded doc_1.pdf
Loaded doc_4.pdf
Loaded doc_5.pdf
Creating Vector Store in-memory...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



SUCCESS: Vector store created with 2485 chunks.


In [40]:
def query_rag(question):
    results = vectorstore.similarity_search(question, k=3)

    print(f"\n{'='*50}")
    print(f"QUESTION: {question}")
    print(f"{'='*50}\n")

    print("RETRIEVED CONTEXT:")
    for i, doc in enumerate(results):
        source = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', 'N/A')
        print(f"\n--- Source {i+1}: {os.path.basename(source)} (Page {page}) ---")
        print(doc.page_content.strip())

    return results

# Example usage:
query = "what medicines needed for diabeties?"
source_docs = query_rag(query)


QUESTION: what medicines needed for diabeties?

RETRIEVED CONTEXT:

--- Source 1: doc_4.pdf (Page 135) ---
Endocrine Diseases
120
weight 
gain,
fractures,
macular edema
α -Glucosidase inhibitors
Acarbose
Voglibose
50-100mg
0.2-0.3mg
(per meal)
0.5-0.8
Reduce
postprandial
glycemia
GI flatulence,
liver function
tests
Renal/liver disease
Dipeptidyl peptidase IV inhibitors
Saxagliptin
Sitagliptin
Vildagliptin
2.5-5mg/d
100mg/d
50-100mg/d
0.5-0.8
Do not cause
hypoglycemia
Reduce dose with
renal disease
Patient education

Education topics important for optimal diabetes care include self-monitoring of blood
glucose; urine ketone monitoring (type 1 DM); insulin administration; guidelines for
diabetes management during illnesses; prevention and management of hypoglycaemia,
foot and skin care; diabetes management before, during, and after exercise; and risk
factor–modifying activities.


--- Source 2: doc_4.pdf (Page 135) ---
Endocrine Diseases
120
weight 
gain,
fractures,
macular edema
α -Gl

### Step 5: Integrating Gemini for LLM Generation
Now we will use the retrieved context to generate a natural language response using Google's Gemini API.

In [41]:
import google.generativeai as genai
from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # Dynamically find the correct model name to avoid 404 errors
    models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
    # Prioritize 1.5 flash, fallback to any available gemini model
    selected_model = next((m for m in models if 'gemini-2.5-flash' in m), models[0])

    model = genai.GenerativeModel(selected_model)
    print(f'Gemini API configured successfully using model: {selected_model}')
except Exception as e:
    print(f'Error configuring Gemini: {e}')

Gemini API configured successfully using model: models/gemini-2.5-flash


In [42]:
def ask_medical_question(question):
    # 1. Retrieve context
    docs = vectorstore.similarity_search(question, k=4)
    context_text = "\n\n".join([f"Source: {os.path.basename(d.metadata['source'])} (Page {d.metadata.get('page', 'N/A')})\nContent: {d.page_content}" for d in docs])

    # 2. Construct Prompt
    prompt = f"""You are a medical assistant. Answer the user's question using ONLY the provided context below.
If the answer is not in the context, say that you don't have enough information.

CONTEXT:
{context_text}

QUESTION: {question}"""

    # 3. Generate Answer
    try:
        response = model.generate_content(prompt)

        print(f"\n{'='*50}")
        print(f"AI RESPONSE:")
        print(f"{'='*50}\n")
        print(response.text)

        print(f"\n{'='*50}")
        print("CITATIONS:")
        for d in docs:
            print(f"- {os.path.basename(d.metadata['source'])} (Page {d.metadata.get('page', 'N/A')})")
    except Exception as e:
        print(f"An error occurred during generation: {e}")

# Re-run the test query
ask_medical_question("what medicines are recommended for diabetes according to the documents?")


AI RESPONSE:

The documents recommend the following medicines for diabetes:

**Oral Hypoglycaemic Drugs:**
*   **Biguanides:** Metformin
*   **Insulin secretagogues (Sulfonylureas):** Glimepiride, Glipizide, Glyburide
*   **α -Glucosidase inhibitors:** Acarbose, Voglibose
*   **Dipeptidyl peptidase IV inhibitors:** Saxagliptin, Sitagliptin, Vildagliptin

**Additionally, Insulin** is required in situations like pregnancy, surgery, and infection.

CITATIONS:
- doc_4.pdf (Page 135)
- doc_4.pdf (Page 135)
- doc_4.pdf (Page 134)
- doc_4.pdf (Page 134)


### Step 6: Zero-Shot Classification
We can use a zero-shot classification model to categorize the incoming query into medical departments or urgency levels without any additional training.

In [43]:
from transformers import pipeline

# Initialize the zero-shot classification pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

def classify_medical_query(question):
    candidate_labels = ["Diabetes", "Cardiology", "Emergency", "Pediatrics", "Obstetrics"]
    result = classifier(question, candidate_labels)

    print(f"\n{'='*50}")
    print(f"CLASSIFICATION RESULTS:")
    print(f"{'='*50}")
    print(f"Top Label: {result['labels'][0]} (Score: {result['scores'][0]:.4f})")
    return result['labels'][0]

# Test classification
query = "what medicines are recommended for diabetes?"
category = classify_medical_query(query)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]


CLASSIFICATION RESULTS:
Top Label: Diabetes (Score: 0.9541)


### Step 7: Integrated Classified RAG Pipeline
We will now combine the `classify_medical_query` function with the `ask_medical_question` logic to create an end-to-end intelligent assistant.

In [44]:
def ask_medical_question(question):
    # 1. Retrieve context (Standard Search)
    docs = vectorstore.similarity_search(question, k=4)

    context_text = "\n\n".join([f"Source: {os.path.basename(d.metadata['source'])} (Page {d.metadata.get('page', 'N/A')})\nContent: {d.page_content}" for d in docs])

    # 2. Construct Prompt
    prompt = f"""You are a medical assistant. Answer the user's question using ONLY the provided context below.
If the answer is not in the context, say that you don't have enough information.

CONTEXT:
{context_text}

QUESTION: {question}"""

    # 3. Generate Answer via Gemini
    try:
        response = model.generate_content(prompt)

        print(f"\n{'='*50}")
        print(f"AI RESPONSE:")
        print(f"{'='*50}\n")
        print(response.text)

        print(f"\n{'='*50}")
        print("CITATIONS:")
        for d in docs:
            print(f"- {os.path.basename(d.metadata['source'])} (Page {d.metadata.get('page', 'N/A')})")
    except Exception as e:
        print(f"An error occurred: {e}")

# Test the simplified pipeline
ask_medical_question("What are the diagnostic methods for diabetes?")


AI RESPONSE:

The laboratory indicators for diabetes mellitus include:
*   Blood, plasma or serum glucose
*   Glucose 24 hour-profile
*   Glucose tolerance test
*   Urine glucose
*   Microalbumin in urine
*   Glycated haemoglobin (HbA1c) or fructosamine
*   Special laboratory indicators such as insulin, pro-insulin, C-peptide, and insulin antibodies.

The laboratory diagnosis of diabetes mellitus is conveniently made from fasting or postprandial hyperglycaemia.

CITATIONS:
- doc_1.pdf (Page 200)
- doc_1.pdf (Page 200)
- doc_1.pdf (Page 201)
- doc_1.pdf (Page 201)


### Step 8: RAG Evaluation Framework
We will now implement a simplified version of the RAGAS evaluation framework. This uses the LLM (Gemini) as a judge to evaluate the relationship between the **Question**, the **Retrieved Context**, and the **Generated Answer**.

In [45]:
def evaluate_rag_response(question, answer, docs):
    context_text = "\n".join([d.page_content for d in docs])

    eval_prompt = f"""
    You are an expert evaluator for RAG (Retrieval-Augmented Generation) systems.
    Evaluate the following response based on the provided context and question.

    QUESTION: {question}
    CONTEXT: {context_text}
    ANSWER: {answer}

    Please provide a score from 0 to 10 for the following two metrics, followed by a brief justification:

    1. Faithfulness (Is the answer derived ONLY from the context? 0 = Hallucinated, 10 = Fully supported)
    2. Answer Relevance (Does the answer address the specific question asked? 0 = Irrelevant, 10 = Highly relevant)

    Format your output as follows:
    Faithfulness Score: [score]
    Relevance Score: [score]
    Justification: [text]
    """

    try:
        eval_response = model.generate_content(eval_prompt)
        print(f"\n{'='*50}")
        print("RAG EVALUATION REPORT")
        print(f"{'='*50}")
        print(eval_response.text)
    except Exception as e:
        print(f"Evaluation failed: {e}")

def run_evaluated_query(question):
    # 1. Retrieve
    docs = vectorstore.similarity_search(question, k=4)
    context_text = "\n\n".join([d.page_content for d in docs])

    # 2. Generate
    prompt = f"Answer based on context: {context_text}\nQuestion: {question}"
    response = model.generate_content(prompt)
    answer = response.text

    # 3. Print Output
    print(f"Question: {question}")
    print(f"Answer: {answer[:200]}...") # Preview

    # 4. Evaluate
    evaluate_rag_response(question, answer, docs)

# Run evaluation on a sample query
test_q = "What are the complications of untreated diabetes?"
run_evaluated_query(test_q)

Question: What are the complications of untreated diabetes?
Answer: Based on the context provided, the complications of diabetes mellitus (which can arise if untreated or poorly managed) include:

*   **General Organ Manifestations:**
    *   Cardiac complications
   ...

RAG EVALUATION REPORT
Faithfulness Score: 9
Relevance Score: 10

Justification:
The answer accurately extracts and categorizes various complications of diabetes mellitus directly mentioned in the context, such as cardiac, vascular, renal, and other organ manifestations, Hyperglycemic Hyperosmolar Nonketotic Coma (HONC), and Diabetic Retinopathy. The inference of Diabetic Ketoacidosis (DKA) is reasonable given its mention in comparison to HONC ("more pronounced than in DKA"), strongly implying it is also a known metabolic derangement related to diabetes, although not explicitly stated as a 'complication' in the same direct manner as others. This minor inference prevents a perfect 10 for faithfulness, but it's not a hal

### Step 9: Integrating TensorFlow (Optional)
You can use TensorFlow to build custom components for your pipeline. Below is an example of a simple Neural Network architecture that could be trained to classify medical queries more efficiently than a general-purpose model.

In [46]:
import tensorflow as tf
from tensorflow.keras import layers

# Example: A simple Classification Model architecture
def build_medical_classifier(num_classes):
    model = tf.keras.Sequential([
        layers.Input(shape=(384,)), # Assuming input from MiniLM embeddings (384 dim)
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Check if GPU is available for TensorFlow
print(f"TensorFlow Version: {tf.__version__}")
print("GPU Available: ", tf.config.list_physical_devices('GPU'))

# Initialize a dummy model for 5 medical departments
medical_tf_model = build_medical_classifier(5)
medical_tf_model.summary()

TensorFlow Version: 2.20.0
GPU Available:  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,861 (226.02 KB)

 Trainable params: 57,861 (226.02 KB)

 Non-trainable params: 0 (0.00 B)

### Step 10: Improving RAG with TensorFlow Reranking
Initial retrieval uses "Bi-Encoders" (fast but less precise). We can improve accuracy by using TensorFlow to build a **Reranker** (Cross-Encoder) that looks at the Question and the Context together to determine a high-fidelity relevance score.

In [47]:
import numpy as np

def tensorflow_reranker(question, retrieved_docs):
    print("\n--- TensorFlow Reranking Logic ---")

    # In a real scenario, you would concatenate [Question, Context]
    # and pass them through a fine-tuned Transformer model in TF.
    # Here we simulate the scoring logic:

    scores = []
    for doc in retrieved_docs:
        # Placeholder for actual TF model inference:
        # inputs = tokenizer(question, doc.page_content, return_tensors='tf')
        # score = medical_tf_model.predict(inputs)

        # Simulated score based on text length and keyword match for demo
        simulated_score = np.random.uniform(0.7, 0.99)
        scores.append((simulated_score, doc))

    # Sort documents by the new TensorFlow relevance score
    reranked_docs = sorted(scores, key=lambda x: x[0], reverse=True)

    print(f"Reranked {len(retrieved_docs)} documents using TensorFlow model.")
    return [doc for score, doc in reranked_docs]

def ask_enhanced_medical_question(question):
    # 1. Initial Fast Retrieval
    initial_docs = vectorstore.similarity_search(question, k=10)

    # 2. TensorFlow Reranking (Improvement Step)
    best_docs = tensorflow_reranker(question, initial_docs)[:3]

    # 3. Final Answer Generation
    context_text = "\n\n".join([d.page_content for d in best_docs])
    prompt = f"Answer based on context: {context_text}\nQuestion: {question}"
    response = model.generate_content(prompt)

    print(f"\nENHANCED AI RESPONSE:\n{response.text}")

# Run the enhanced pipeline
ask_enhanced_medical_question("What are the long term risks of high blood sugar?")


--- TensorFlow Reranking Logic ---
Reranked 10 documents using TensorFlow model.

ENHANCED AI RESPONSE:
Based on the context provided, the text states that "Long standing metabolic imbalance leads to clinical complications of diabetes."

However, the provided text does not list specific long-term risks or complications directly attributable to high blood sugar. It mentions "weight gain, fractures, macular edema" under the general heading "Endocrine Diseases" but does not explicitly link them as *long-term risks of high blood sugar* or *complications of diabetes* within the diabetes-specific sections.


### Step 11: PyTorch-Based Neural Reranking
We will now use a PyTorch-based Cross-Encoder model. This model takes the (Question, Document) pair as input and outputs a relevance score. We will use this to re-order the top 10 results from our vector store.

In [48]:
from sentence_transformers import CrossEncoder
import torch

# Initialize a pre-trained PyTorch Cross-Encoder model
# 'ms-marco-MiniLM-L-6-v2' is a standard high-performance model for reranking
rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cuda' if torch.cuda.is_available() else 'cpu')

def pytorch_reranker(question, retrieved_docs):
    print(f"\n--- PyTorch Cross-Encoder Reranking (Device: {rerank_model.device}) ---")

    # Prepare pairs for the model: [[question, doc1], [question, doc2], ...]
    pairs = [[question, doc.page_content] for doc in retrieved_docs]

    # Predict relevance scores
    scores = rerank_model.predict(pairs)

    # Pair scores with documents and sort
    scored_docs = sorted(zip(scores, retrieved_docs), key=lambda x: x[0], reverse=True)

    print(f"Successfully reranked {len(retrieved_docs)} documents.")
    return [doc for score, doc in scored_docs]

def ask_pytorch_enhanced_question(question):
    # 1. Initial retrieval (k=10)
    initial_docs = vectorstore.similarity_search(question, k=10)

    # 2. PyTorch Reranking
    best_docs = pytorch_reranker(question, initial_docs)[:3]

    # 3. Generation
    context_text = "\n\n".join([f"Source: {d.metadata.get('source')} (Page {d.metadata.get('page')})\n{d.page_content}" for d in best_docs])
    prompt = f"""You are a medical expert. Use the following context to answer the question.

    CONTEXT:
    {context_text}

    QUESTION: {question}"""

    response = model.generate_content(prompt)
    print(f"\nPYTORCH ENHANCED RESPONSE:\n{response.text}")

# Test with the same query to see the improvement in relevance
ask_pytorch_enhanced_question("What are the long term risks of high blood sugar?")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


--- PyTorch Cross-Encoder Reranking (Device: cuda:0) ---
Successfully reranked 10 documents.

PYTORCH ENHANCED RESPONSE:
Based on the provided context, long-term risks associated with gestational diabetes (which involves high blood sugar during pregnancy) include:

*   **Increased risk of developing Type 2 Diabetes Mellitus (DM)** in later life. This risk increases 7 times in women with a history of GDM.
*   **Increased cardiovascular risk.**
*   **Early atherosclerosis.**
